# 1. Setup
**Approach 2: Feature disentanglement.** The backbone feature is split into a crop part and a disease part.
Adversarial heads (with gradient reversal) try to read the disease from the crop part, and the crop
from the disease part. The encoders learn to defeat them, so each part keeps only its own concept.
An orthogonality loss also stops the two parts from carrying the same information.
Idea: if disease features hold no crop information, a seen disease is easier to spot on a new crop.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F

# Find the project root
ROOT = Path.cwd()
if not (ROOT / "common").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from common.config import CHECKPOINT_DIR, RESULTS_DIR, DEVICE, NUM_CROPS, NUM_DISEASES, set_seed
from common.dataset import get_loaders, get_pairs
from common.evaluate import evaluate, val_accuracy

# Use True for a quick test
DEBUG = True

# Shared training settings
HP = {
    "backbone": "resnet50",
    "lr": 1e-4,
    "weight_decay": 1e-4,
    "epochs": 20,
    "batch_size": 16 if DEBUG else 64,
    "concept_dim": 512,
    "label_smoothing": 0.1
}

# Experiment name, orthogonal weight, adversarial weight
EXPERIMENTS = [
    ("exp1_encoders", 0.0, 0.0),
    ("exp2_orthogonal", 0.1, 0.0),
    ("exp3_adversarial", 0.0, 0.3),
    ("exp4_combined", 0.1, 0.3)
]

EPOCHS = 1 if DEBUG else HP["epochs"]
pairs = get_pairs().to(DEVICE)

print("Device:", DEVICE)
print("Epochs:", EPOCHS)


# 2. Data

In [8]:
train_loader, val_loader, seen_loader, unseen_loader = get_loaders(
    HP["batch_size"], norm="imagenet", debug=DEBUG)
pairs = get_pairs().to(DEVICE)

# 3. Model

In [ ]:

class GradientReversal(torch.autograd.Function):

    # Keep features unchanged during forward pass
    @staticmethod
    def forward(ctx, x):
        return x.view_as(x)

    # Reverse gradients during backward pass
    @staticmethod
    def backward(ctx, grad):
        return -grad


class DisentangleNet(nn.Module):

    def __init__(self):
        super().__init__()

        # Extract shared image features
        self.backbone = timm.create_model(
            HP["backbone"], pretrained=True, num_classes=0
        )

        width = self.backbone.num_features
        dim = HP["concept_dim"]

        # Learn crop and disease features separately
        self.crop_encoder = nn.Sequential(
            nn.Linear(width, dim),
            nn.ReLU()
        )

        self.disease_encoder = nn.Sequential(
            nn.Linear(width, dim),
            nn.ReLU()
        )

        # Predict crop and disease classes
        self.crop_head = nn.Linear(dim, NUM_CROPS)
        self.disease_head = nn.Linear(dim, NUM_DISEASES)

        # Predict the opposite task
        self.disease_from_crop = nn.Linear(dim, NUM_DISEASES)
        self.crop_from_disease = nn.Linear(dim, NUM_CROPS)

    def forward(self, images, use_adversary=False):
        shared = self.backbone(images)

        crop = self.crop_encoder(shared)
        disease = self.disease_encoder(shared)

        logits = (
            self.crop_head(crop),
            self.disease_head(disease)
        )

        adversaries = None

        if use_adversary:
            adversaries = (
                self.disease_from_crop(GradientReversal.apply(crop)),
                self.crop_from_disease(GradientReversal.apply(disease))
            )

        return logits, adversaries, (crop, disease)


# Reduce correlation between crop and disease features
def orthogonal_loss(crop, disease):
    crop = F.normalize(crop - crop.mean(0), dim=0)
    disease = F.normalize(disease - disease.mean(0), dim=0)

    return (crop.T @ disease).square().mean()


# 4. Prediction and Feature Analysis

In [ ]:
def make_predict(model):

    @torch.no_grad()
    def predict(images):
        model.eval()

        (crop, disease), _, _ = model(images.to(DEVICE))

        crop = F.log_softmax(crop.float(), dim=1)
        disease = F.log_softmax(disease.float(), dim=1)

        # Combine crop and disease scores
        return crop[:, pairs[:, 0]] + disease[:, pairs[:, 1]]

    return predict


@torch.no_grad()
def measure_correlation(model, loader):
    model.eval()

    crop_list = []
    disease_list = []

    # Collect features from validation images
    for images, _, _, _ in loader:
        _, _, (crop, disease) = model(images.to(DEVICE))

        crop_list.append(crop.float().cpu())
        disease_list.append(disease.float().cpu())

    crop_features = torch.cat(crop_list)
    disease_features = torch.cat(disease_list)

    return orthogonal_loss(crop_features, disease_features).item()


epoch 1: CE 9.133, adversary 9.156, orthogonal 0.1155, val top-1 4.00%


<All keys matched successfully>

# 5. Training All Four Experiments

In [ ]:
CHECKPOINT_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

records = []

for name, orth_w, adv_w in EXPERIMENTS:

    # Start each experiment with the same seed
    set_seed(42)

    train_loader, val_loader, _, _ = get_loaders(
        HP["batch_size"], "imagenet", DEBUG
    )

    model = DisentangleNet().to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=HP["lr"],
        weight_decay=HP["weight_decay"]
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS
    )

    criterion = nn.CrossEntropyLoss(
        label_smoothing=HP["label_smoothing"]
    )

    exp_hp = {
        **HP,
        "orthogonal_weight": orth_w,
        "adversarial_weight": adv_w
    }

    path = CHECKPOINT_DIR / f"approach2_{name}.pt"
    best_val = -1.0
    predict = make_predict(model)

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0

        for images, crop_y, disease_y, _ in train_loader:
            images = images.to(DEVICE)
            crop_y = crop_y.to(DEVICE)
            disease_y = disease_y.to(DEVICE)

            # Forward pass
            (crop_logits, disease_logits), adv, (crop_f, disease_f) = model(
                images, use_adversary=adv_w > 0
            )

            # Classification loss
            ce = (
                criterion(crop_logits, crop_y)
                + criterion(disease_logits, disease_y)
            )

            # Adversarial loss
            adv_loss = (
                criterion(adv[0], disease_y)
                + criterion(adv[1], crop_y)
                if adv_w > 0
                else torch.zeros((), device=DEVICE)
            )

            # Orthogonality loss
            orth_loss = (
                orthogonal_loss(crop_f.float(), disease_f.float())
                if orth_w > 0
                else torch.zeros((), device=DEVICE)
            )

            # Total training loss
            loss = ce + adv_w * adv_loss + orth_w * orth_loss

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        scheduler.step()

        # Use validation to select the best epoch
        score = val_accuracy(predict, val_loader)

        if score > best_val:
            best_val = score

            torch.save({
                "model_name": name,
                "state_dict": {
                    k: v.detach().cpu()
                    for k, v in model.state_dict().items()
                },
                "hp": exp_hp
            }, path)

        print(
            f"{name} | Epoch {epoch + 1}/{EPOCHS} | "
            f"Loss {total_loss / len(train_loader):.3f} | "
            f"Val Top-1 {score:.2f}%"
        )

    # Restore the best epoch
    saved = torch.load(path, map_location="cpu", weights_only=True)
    model.load_state_dict(saved["state_dict"])

    correlation = measure_correlation(model, val_loader)

    records.append({
        "experiment": name,
        "orth_weight": orth_w,
        "adv_weight": adv_w,
        "val_top1": best_val,
        "val_correlation": correlation
    })

    del predict, model, optimizer, scheduler

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


{
  "model": "approach2_disentangle",
  "seen_top1": 0.0,
  "seen_top5": 6.0,
  "unseen_top1": 0.0,
  "unseen_top5": 0.0,
  "harmonic_mean": 0.0,
  "crop_acc": 3.0,
  "disease_acc": 19.0,
  "robustness": {
    "brightness": {
      "top1": 1.0,
      "drop": -1.0
    },
    "blur": {
      "top1": 1.0,
      "drop": -1.0
    },
    "noise": {
      "top1": 1.0,
      "drop": -1.0
    }
  }
}


# 6. Compare Experiments

In [ ]:
# Compare the four experiments
summary = pd.DataFrame(records)
summary = summary.sort_values("val_top1", ascending=False)

summary.to_csv(
    RESULTS_DIR / "approach2_validation.csv",
    index=False
)

display(summary)

winner = summary.iloc[0]["experiment"]

print("Best validation experiment:", winner)


# 7. Final Evaluation and Save

In [ ]:
if DEBUG:
    print("Debug complete")
    print("Set DEBUG to False for full training and final evaluation")

else:
    # Load the best validation model
    saved = torch.load(
        CHECKPOINT_DIR / f"approach2_{winner}.pt",
        map_location="cpu",
        weights_only=True
    )

    model = DisentangleNet().to(DEVICE)
    model.load_state_dict(saved["state_dict"])

    predict = make_predict(model)

    _, _, seen_loader, unseen_loader = get_loaders(
        HP["batch_size"], "imagenet", False
    )

    # Run the official team evaluation
    results = evaluate(
        predict,
        seen_loader,
        unseen_loader,
        "approach2_disentangle",
        saved["hp"],
        robust=True
    )

    # Save the selected model for team integration
    torch.save({
        "model_name": "approach2_disentangle",
        "state_dict": model.state_dict(),
        "hp": saved["hp"]
    }, CHECKPOINT_DIR / "approach2_disentangle.pt")
